# 🧬 Tesis Maestría: Ejecución de Experimentos en Google Colab con GPU

Este cuaderno Jupyter está optimizado para guiarte en el entrenamiento y la validación de los modelos generativos oncológicos utilizando la GPU de Google Colab (T4 o A100).

### Estructura recomendada en Drive:
Asegúrate de que la carpeta completa de tu proyecto (`datasets`) esté en la raíz de tu Drive bajo la ruta:
`/MyDrive/tesis_genetica/`

## Paso 1: Montar Google Drive e ir al directorio de trabajo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Cambiar de directorio a la carpeta del proyecto en Drive
%cd /content/drive/MyDrive/datasets/

Mounted at /content/drive
/content/drive/MyDrive/datasets


## Paso 2: Instalar Dependencias del Entorno

In [ ]:
!pip install xgboost joblib pandas pyarrow ctgan tabpfn matplotlib seaborn scikit-learn tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.2/753.2 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 371.2/371.2 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## Paso 3: Limpiar Checkpoints Anteriores (Seguridad)
Es vital eliminar cualquier checkpoint piloto anterior para que las dimensiones (1,001 características, incluyendo el Category) no generen conflictos.

In [ ]:
!rm -rf datasets/results/checkpoints/steps_elite_borda/
!rm -f datasets/results/checkpoints/fd_shell_elite_borda.joblib

## Paso 4: Entrenar el Modelo SOTA (Forest Diffusion)
Este comando entrena los 50 pasos de difusión en paralelo para la cohorte Élite Borda.

In [ ]:
!python -u scripts/train_fd_colab.py \
    --dataset elite_borda \
    --n_t 50 \
    --device cuda \
    --mode train \
    --data_subpath datasets/results/elite_borda_training_table.parquet \
    --checkpoint_subpath datasets/results/checkpoints


  Forest Diffusion — Colab GPU Training
  Modo:       TRAIN
  n_t:        50 pasos
  Dispositivo:CUDA
  Drive base: /content/drive/MyDrive
  Input:      /content/drive/MyDrive/datasets/results/elite_borda_training_table.parquet
  Steps dir:  /content/drive/MyDrive/datasets/results/checkpoints/steps_elite_borda

📂 Cargando dataset desde Drive...
   ✅ 9,309 muestras × 1,002 features

🔄 Shell detectado — reanudando sesión previa
   ✅ Pasos completados: 26/50
   ▶️  Reanudando desde el paso 27

🧬 Iniciando entrenamiento (1002 XGBoosts/paso en paralelo)

⏳ [Paso 27/50] Entrenando 1002 XGBoosts...
   ✅ 14.9 min | 737 MB en Drive | ETA restante: 5.7h
⏳ [Paso 28/50] Entrenando 1002 XGBoosts...
   ✅ 14.8 min | 737 MB en Drive | ETA restante: 5.4h
⏳ [Paso 29/50] Entrenando 1002 XGBoosts...
   ✅ 14.8 min | 737 MB en Drive | ETA restante: 5.2h
⏳ [Paso 30/50] Entrenando 1002 XGBoosts...
   ✅ 14.5 min | 738 MB en Drive | ETA restante: 4.9h
⏳ [Paso 31/50] Entrenando 1002 XGBoosts...
   ✅ 14.6 min | 

In [ ]:
!python scripts/train_fd_colab.py --mode assemble


🚀 GPU disponible: |   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |

  Forest Diffusion — Colab GPU Training
  Modo:       ASSEMBLE
  n_t:        50 pasos
  Dispositivo:CUDA
  Drive base: /content/drive/MyDrive
  Input:      /content/drive/MyDrive/datasets/results/elite_borda_training_table.parquet
  Steps dir:  /content/drive/MyDrive/datasets/results/checkpoints/steps_elite_borda


🔍 Pasos encontrados: 50/50
✅ Todos los pasos presentes. Ensamblando modelo final...
   📦 Destino: /content/drive/MyDrive/datasets/results/checkpoints/forest_diffusion_model_elite_borda.joblib

   📥 [50/50] cargado — ETA restante: 0.0 min    

💾 Guardando modelo maestro (compress=3, puede tardar varios minutos)...

🎉 ¡ENSAMBLAJE COMPLETADO en 30.6 min!
📦 Tamaño final: 13.0 GB
📂 Ruta: /content/drive/MyDrive/datasets/results/checkpoints/forest_diffusion_model_elite_borda.joblib


## Paso 5: Entrenar el Modelo de Línea de Base (CTGAN)
Entrenamos el modelo generativo adversario clásico por 100 épocas en GPU.

In [ ]:
!python -u scripts/train_ctgan_colab.py \
    --epochs 100 \
    --batch_size 500 \
    --data_subpath datasets/results/elite_borda_training_table.parquet \
    --output_dir datasets/results

  CTGAN — Colab GPU Training
  Épocas:      100
  Batch Size:  500
  Input:      datasets/results/elite_borda_training_table.parquet
📁 Google Drive detectado como almacenamiento base.
✅ Dataset cargado: 9309 muestras × 1003 columnas
🧬 Columnas a entrenar: 1002 (Discretas: ['Category', 'Technology_Label'])
🖥️  ¿GPU disponible para PyTorch?: SÍ (CUDA)
/usr/local/lib/python3.12/dist-packages/ctgan/synthesizers/_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(

🚀 Iniciando entrenamiento del modelo CTGAN...
Gen. (+00.00) | Discrim. (+00.00):   0% 0/100 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls in

## Paso 6: Generación de 120,000 Muestras Sintéticas Definitivas (Forest Diffusion)
Utiliza la integración de Euler optimizada para generar la base sintética masiva de Forest Diffusion.

In [ ]:
!python -u scripts/generate_low_memory.py \
    --dataset elite_borda \
    --n_samples 120000 \
    --n_t 50

Streaming output truncated to the last 5000 lines.

  return func(**kwargs)
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [11:15:52] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "predictor" } are not used.

  return func(**kwargs)
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [11:15:53] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "predictor" } are not used.

  return func(**kwargs)
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [11:15:54] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "predictor" } are not used.

  return func(**kwargs)
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [11:15:55] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "predictor" } are not used.

  return func(**kwargs)
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [11:15:56] WARNING: /__w/xgboost/xgboo

In [ ]:
!cp results/checkpoints/forest_diffusion_model_elite_borda.joblib results/forest_diffusion_model_elite_borda.joblib


In [ ]:
!pip install ForestDiffusion


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 20.0 MB/s eta 0:00:00


In [ ]:
!python scripts/generate_forest_samples.py --dataset elite_borda --n_samples 120000



🌲 Iniciando Generación SOTA 2026: Forest Diffusion
  Dataset:  elite_borda
  Muestras: 120000

📖 Leyendo nombres de genes desde results/elite_borda_training_table.parquet...
   ✅ Se identificaron 1002 características biológicas.

🧠 Cargando modelo ensamblado (results/forest_diffusion_model_elite_borda.joblib)...
   🧬 Herencia de ForestDiffusionModel asignada dinámicamente.
   🔧 Convirtiendo XGBRegressors a Boosters nativos para generación rápida...
   ✅ Modelo cargado en 533.4 segundos.

🚀 Generando 120000 pacientes sintéticos...
Traceback (most recent call last):
  File "/content/drive/MyDrive/datasets/scripts/generate_forest_samples.py", line 125, in <module>
    generate_samples()
  File "/content/drive/MyDrive/datasets/scripts/generate_forest_samples.py", line 101, in generate_samples
    synthetic_numpy = model.generate(batch_size=args.n_samples)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ForestDiffusion/diffusi

In [ ]:
!python scripts/generate_forest_samples_gpu.py --dataset elite_borda --n_samples 5000


python3: can't open file '/content/scripts/generate_forest_samples_gpu.py': [Errno 2] No such file or directory


In [ ]:
!python scripts/generate_forest_samples_gpu.py --dataset elite_borda --n_samples 120000



🌲 Iniciando Generación Optimizada GPU (Plan B)
  Dataset:  elite_borda
  Muestras: 120000

📖 Leyendo nombres de genes desde results/elite_borda_training_table.parquet...
   ✅ Se identificaron 1002 características biológicas.

🧠 Cargando modelo ensamblado (results/forest_diffusion_model_elite_borda.joblib)...
   🧬 Herencia de ForestDiffusionModel asignada dinámicamente.
   🔧 Convirtiendo XGBRegressors a Boosters nativos y configurando CUDA...
   ✅ Modelo cargado en 546.4 segundos.

🎲 Inicializando 120000 pacientes con ruido puro en GPU (CuPy)...

🚀 Iniciando Des-Ruidificación (50 pasos en GPU A100)...
Pasos de Difusión: 100% 50/50 [11:52<00:00, 14.25s/it]
✅ Generación completada con éxito en 11.88 minutos.

🧬 Revirtiendo escalado matemático a perfiles genómicos...

💾 Muestras biológicas guardadas en: results/synthetic_samples_elite_borda_120000.parquet


## Paso 7: Generación de 120,000 Muestras Sintéticas Definitivas (CTGAN)
Generamos las muestras sintéticas para el modelo clásico adversario.

In [ ]:
import joblib
import pandas as pd
from tqdm import tqdm

print("🎲 Generando muestras con CTGAN por lotes...")
model_path = 'results/ctgan_model_elite_borda.pkl'
output_path = 'results/synthetic_samples_ctgan_elite_borda_120000.parquet'

model = joblib.load(model_path)

# Generar en 6 lotes de 20,000 para evitar picos de memoria en GPU/RAM
chunks = []
for i in tqdm(range(6), desc="Lotes Generados"):
    chunks.append(model.sample(20000))

synthetic_df = pd.concat(chunks, ignore_index=True)

# OPTIMIZACIÓN PREVENTIVA: Ajustar categorías discretas a enteros biológicos
for col in ['Technology_Label', 'Category', 'target']:
    if col in synthetic_df.columns:
        synthetic_df[col] = synthetic_df[col].round().clip(0, 1).astype(int)

synthetic_df.to_parquet(output_path)
print(f"✅ Guardado en {output_path}")


🎲 Generando muestras con CTGAN por lotes...


Lotes Generados: 100%|██████████| 6/6 [01:33<00:00, 15.65s/it]


✅ Guardado en results/synthetic_samples_ctgan_elite_borda_120000.parquet


## Paso 8: Explicabilidad Post-Entrenamiento (Firma Jaccard)
Calcula el solapamiento Jaccard de los biomarcadores entre la data real y la sintética definitiva.

In [ ]:
!mkdir -p results/metrics


In [ ]:
!python scripts/analyze_borda_feature_importance.py \
    --real_path results/elite_borda_training_table.parquet \
    --synth_path results/synthetic_samples_elite_borda_120000.parquet


📂 Cargando datos...
Real: results/elite_borda_training_table.parquet
Synth: results/synthetic_samples_elite_borda_120000.parquet
🌲 Analizando importancia en datos reales...
🌲 Analizando importancia en datos sintéticos...

=== RESULTADOS DE FIDELIDAD DE BIOMARCADORES (Borda) ===
Jaccard Similarity (Top 50): 0.3158
Número de genes coincidentes: 24
Genes coincidentes: ['ABLIM1', 'CAV1', 'CCL23', 'CCR4', 'CD36', 'CDC6', 'CKMT2', 'CSF2', 'CXCL11', 'CXCL9', 'EDN1', 'EMP2', 'EPAS1', 'ESM1', 'FOXP3', 'GIMAP6', 'IL4', 'KLF4', 'MFAP4', 'PDCD1LG2', 'SDHA', 'SHCBP1', 'THBD', 'VTCN1']

🏆 Top Biomarcadores Preservados por la IA:
        Gene  Importance_Real  Importance_Synth  Mean_Importance
0       SDHA         0.031563          0.001409         0.016486
13      KLF4         0.005897          0.008725         0.007311
10      CDC6         0.006576          0.006838         0.006707
15      ESM1         0.005164          0.004218         0.004691
6      CKMT2         0.007723          0.001375     

## Paso 9: Validación de la Hipótesis de Edwin
Ejecuta el experimento de control cruzado en las 5 cohortes utilizando los 120k datos sintéticos definitivos.

In [ ]:
!python scripts/evaluate_edwin_hypothesis.py \
    --real_path results/elite_borda_training_table.parquet \
    --synth_path results/synthetic_samples_elite_borda_120000.parquet

🧬 Estudios seleccionados para la validación: ['GSE108712', 'GSE45498', 'GSE150615', 'GSE100150', 'GSE87211']

🧪 Evaluando: GSE108712
    [Escaso]  Local: 0.982 | Aumentado (Sintético): 0.976 | Sinergia Global (Real): 0.981
    [Completo] Local: 0.988

🧪 Evaluando: GSE45498
    [Escaso]  Local: 0.597 | Aumentado (Sintético): 0.605 | Sinergia Global (Real): 0.709
    [Completo] Local: 0.731

🧪 Evaluando: GSE150615
    [Escaso]  Local: 0.739 | Aumentado (Sintético): 0.833 | Sinergia Global (Real): 0.923
    [Completo] Local: 0.871

🧪 Evaluando: GSE100150
    [Escaso]  Local: 0.881 | Aumentado (Sintético): 0.911 | Sinergia Global (Real): 0.959
    [Completo] Local: 0.996

🧪 Evaluando: GSE87211
    [Escaso]  Local: 0.925 | Aumentado (Sintético): 0.997 | Sinergia Global (Real): 0.981
    [Completo] Local: 0.990

💾 Resultados guardados en results/metrics/edwin_hypothesis_results.csv
🎨 Gráfica guardada en results/metrics/validacion_hipothesis_edwin.png


## Paso 10: Benchmark SOTA de Clasificadores (TSTR)
Entrenamos XGBoost, CatBoost, TabPFN, SVM y RF con la data generada y evaluamos contra pacientes reales.

In [ ]:
!python scripts/run_modern_benchmark_elite.py \
    --real_path results/elite_borda_training_table.parquet \
    --synth_path results/synthetic_samples_elite_borda_120000.parquet

🖥️  Dispositivo detectado para TabPFN: CUDA

🚀 Iniciando Arena SOTA ÉLITE (Borda): Real_Elite_Borda_2026
   (Frontera Tecnológica: SHAP Selector + TabPFN Transformer)
🧬 Generando Selección con: SHAP
🔍 Aplicando Selección de Atributos: shap (Top 100 genes)...
   (SHAP: Entrenando modelo de referencia para explicar importancia...)
✅ Selección completada: 100 genes identificados.
   ⏱️ Eval: shap + XGBoost...
   ⏱️ Eval: shap + CatBoost...
   ⏱️ Eval: shap + TabPFN...

TabPFN requires a one-time license acceptance to download model weights for local inference.

No display detected. Open this URL in a browser on another device:

  https://ux.priorlabs.ai/login?hf_repo_id=tabpfn_3

After logging in, accept the license on the Licenses tab,
then copy your API Key from
  https://ux.priorlabs.ai/account

  [c] Copy URL to clipboard    Paste your API key to continue

> tabpfn_sk_JcD90H6GYdROxUlyCpGvXBfkOl-EnO8mbbFAVKQSfPw
License accepted — API key cached for future sessions.

tabpfn-v3-classifi

In [ ]:
!python scripts/run_modern_benchmark_elite_cpu.py \
    --real_path results/elite_borda_training_table.parquet \
    --synth_path results/synthetic_samples_elite_borda_120000.parquet


🖥️  Dispositivo detectado para TabPFN: CPU

🚀 Iniciando Arena SOTA ÉLITE (Borda): Real_Elite_Borda_2026
   (Frontera Tecnológica: SHAP Selector + TabPFN Transformer)
   ⚖️ Optimización CPU: Muestreando 5000 pacientes
🧬 Generando Selección con: SHAP
🔍 Aplicando Selección de Atributos: shap (Top 100 genes)...
   (SHAP: Entrenando modelo de referencia para explicar importancia...)
✅ Selección completada: 100 genes identificados.
   ⏱️ Eval: shap + XGBoost...
   ⏱️ Eval: shap + CatBoost...
   ⏱️ Eval: shap + TabPFN...

TabPFN requires a one-time license acceptance to download model weights for local inference.

No display detected. Open this URL in a browser on another device:

  https://ux.priorlabs.ai/login?hf_repo_id=tabpfn_3

After logging in, accept the license on the Licenses tab,
then copy your API Key from
  https://ux.priorlabs.ai/account

  [c] Copy URL to clipboard    Paste your API key to continue

> tabpfn_sk_JcD90H6GYdROxUlyCpGvXBfkOl

TabPFN requires a one-time license accep

In [ ]:
# 1. Instalar la librería de mRMR
!pip install mrmr-selection

In [ ]:
# 2. Ejecutar la verificación específica de mRMR
!python scripts/verify_mrmr_colab.py


🖥️ Dispositivo detectado para TabPFN: CPU
📁 Cargando Real desde: results/elite_borda_training_table.parquet
📁 Cargando Sintético desde: results/synthetic_samples_elite_borda_5000.parquet

🚀 Iniciando mRMR en Real (Control) (Seleccionando Top 100 Genes)...
100% 100/100 [00:24<00:00,  4.11it/s]
✅ Genes seleccionados por mRMR. Iniciando evaluación predictiva (3-Fold CV)...
   📊 [mRMR + XGBoost] ROC-AUC: 0.927255 ± 0.000335
   📊 [mRMR + CatBoost] ROC-AUC: 0.915709 ± 0.000591
   📊 [mRMR + TabPFN] ROC-AUC: 0.944204 ± 0.002598
   📊 [mRMR + SVM_RBF_SOTA] ROC-AUC: 0.838600 ± 0.004891
   📊 [mRMR + RandomForest_SOTA] ROC-AUC: 0.930759 ± 0.003878

🚀 Iniciando mRMR en Sintético (TSTS) (Seleccionando Top 100 Genes)...
100% 100/100 [00:23<00:00,  4.21it/s]
✅ Genes seleccionados por mRMR. Iniciando evaluación predictiva (3-Fold CV)...
   📊 [mRMR + XGBoost] ROC-AUC: 0.644192 ± 0.006562
   📊 [mRMR + CatBoost] ROC-AUC: 0.655290 ± 0.008016
   📊 [mRMR + TabPFN] ROC-AUC: 0.678978 ± 0.006230
   📊 [mRMR + SVM

In [ ]:
!python scripts/debug_colab_metadata.py


File: results/synthetic_samples_elite_borda_5000.parquet
  Size: 28637749 bytes
  Modified: Sun Jun 28 14:37:41 2026
  MD5 Hash: 7e8bf9c4b3575b989eaf3934de1d5177
  Shape: (5000, 1002)
  Target counts:
Category
1    3687
0    1313
Name: count, dtype: int64
  SDHA first 5 values:
[3.261688   0.02240208 3.0730135  3.290499   3.23236   ]
